# DistilBERT (starter) — Jigsaw toxic comments

Same layout as `03_bert.ipynb` but **`preprocess_for_distilbert`** and **DistilBERT** weights (fewer parameters, faster).

**Quick iteration:** In the next cell, set `QUICK_ITERATION = True` for a small subset (same idea as `03_bert.ipynb`); `False` uses the full train/validation split.

**Requires:** `pip install torch transformers`

In [1]:
from pathlib import Path
import sys

# Repo root (contains preprocessing/)
ROOT = Path.cwd().resolve()
for c in (ROOT, ROOT.parent):
    if (c / "preprocessing" / "text_preprocessing.py").exists():
        ROOT = c
        break
sys.path.insert(0, str(ROOT))
# notebooks/ for metrics_helpers
sys.path.insert(0, str(ROOT / "notebooks"))

In [2]:
import os
import time

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForSequenceClassification

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

from preprocessing.text_preprocessing import LABEL_COLUMNS, preprocess_for_distilbert
from metrics_helpers import multilabel_evaluation_report, per_label_confusion_matrices, torch_parameter_count

def pick_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda")
    if getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


DEVICE = pick_device()
print("Device:", DEVICE)
torch.manual_seed(42)

QUICK_ITERATION = False

if QUICK_ITERATION:
    MAX_LENGTH = 64
    BATCH_SIZE = 16
    _train_n, _val_n = 512, 256
else:
    MAX_LENGTH = 128
    BATCH_SIZE = 8
    _train_n, _val_n = None, None

EPOCHS = 1
LR = 2e-5

data = preprocess_for_distilbert(
    validation_fraction=0.1,
    random_state=42,
    max_length=MAX_LENGTH,
    return_tensors="pt",
    max_train_samples=_train_n,
    max_val_samples=_val_n,
)


def enc_dict(enc):
    keys = [k for k in ("input_ids", "attention_mask") if k in enc]
    return {k: enc[k] for k in keys}


train_enc = enc_dict(data.train_encodings)
val_enc = enc_dict(data.val_encodings)
y_train = data.y_train
y_val = data.y_val

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(LABEL_COLUMNS),
    problem_type="multi_label_classification",
).to(DEVICE)


class EncDataset(Dataset):
    def __init__(self, enc, labels):
        self.enc = enc
        self.labels = torch.tensor(labels, dtype=torch.float32)

    def __len__(self):
        return self.labels.shape[0]

    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.enc.items()}
        item["labels"] = self.labels[i]
        return item


def collate(batch):
    out = {k: torch.stack([b[k] for b in batch], dim=0) for k in batch[0] if k != "labels"}
    out["labels"] = torch.stack([b["labels"] for b in batch], dim=0)
    return out


train_loader = DataLoader(EncDataset(train_enc, y_train), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate)
val_loader = DataLoader(EncDataset(val_enc, y_val), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate)

opt = torch.optim.AdamW(model.parameters(), lr=LR)
loss_fn = torch.nn.BCEWithLogitsLoss()

print("QUICK_ITERATION:", QUICK_ITERATION, "| train/val rows:", len(y_train), len(y_val), "| max_length:", MAX_LENGTH)

t0 = time.perf_counter()
model.train()
step = 0
for epoch in range(EPOCHS):
    for batch in train_loader:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        labels = batch.pop("labels")
        opt.zero_grad()
        out = model(**batch)
        loss = loss_fn(out.logits, labels)
        loss.backward()
        opt.step()
        step += 1
train_seconds = time.perf_counter() - t0
print(f"Training wall time: {train_seconds:.1f} s, steps: {step}")

Device: mps


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


QUICK_ITERATION: False | train/val rows: 143613 15958 | max_length: 128
Training wall time: 3376.6 s, steps: 17952


In [3]:
model.eval()
all_probs = []
with torch.no_grad():
    for batch in val_loader:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        batch.pop("labels")
        logits = model(**batch).logits
        all_probs.append(torch.sigmoid(logits).cpu().numpy())

prob_val = np.concatenate(all_probs, axis=0)
y_val_np = np.asarray(y_val, dtype=np.int64)
pred_val_05 = (prob_val >= 0.5).astype(np.int64)
print("Validation probabilities ready. Baseline predictions computed at threshold=0.5")

Validation probabilities ready. Baseline predictions computed at threshold=0.5


In [4]:
# Per-label threshold tuning on validation predictions
threshold_grid = np.arange(0.05, 0.951, 0.01)


def binary_f1(y_true, y_pred):
    tp = np.sum((y_true == 1) & (y_pred == 1))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    denom = 2 * tp + fp + fn
    return 0.0 if denom == 0 else (2 * tp) / denom


best_thresholds = {}
threshold_rows = []
pred_val_tuned = np.zeros_like(pred_val_05)

for j, label in enumerate(LABEL_COLUMNS):
    y_true_j = y_val_np[:, j]
    p_j = prob_val[:, j]

    best_t = 0.5
    best_f1 = -1.0

    for t in threshold_grid:
        y_pred_j = (p_j >= t).astype(np.int64)
        f1_j = binary_f1(y_true_j, y_pred_j)
        if f1_j > best_f1:
            best_f1 = f1_j
            best_t = float(t)

    best_thresholds[label] = best_t
    pred_val_tuned[:, j] = (p_j >= best_t).astype(np.int64)
    threshold_rows.append(
        {
            "label": label,
            "best_threshold": round(best_t, 3),
            "best_f1_on_val": best_f1,
        }
    )

threshold_df = pd.DataFrame(threshold_rows)

# Evaluate baseline (0.5) and tuned predictions
per_label_05_df, summary_05 = multilabel_evaluation_report(y_val_np, pred_val_05, prob_val, LABEL_COLUMNS)
per_label_tuned_df, summary_tuned = multilabel_evaluation_report(y_val_np, pred_val_tuned, prob_val, LABEL_COLUMNS)

print("Baseline metrics (threshold=0.5):", summary_05)
print("Tuned metrics (per-label thresholds):", summary_tuned)

summary_compare_df = pd.DataFrame(
    [
        {"setting": "baseline_0.5", **summary_05},
        {"setting": "tuned_per_label", **summary_tuned},
    ]
)

per_label_compare_df = per_label_05_df.merge(
    per_label_tuned_df,
    on="label",
    suffixes=("_baseline", "_tuned"),
).merge(threshold_df[["label", "best_threshold"]], on="label", how="left")

print("\nPer-label chosen thresholds:")
print(best_thresholds)

display(summary_compare_df)
display(threshold_df)
display(per_label_compare_df)

print("\nConfusion matrices (tuned predictions)")
for name, m in per_label_confusion_matrices(y_val_np, pred_val_tuned, LABEL_COLUMNS).items():
    print(name, "\n", m)

print()
print("--- Proposal summary (record for report / comparison) ---")
print(f"  device: {DEVICE}")
print(f"  training_time_s: {train_seconds:.2f}")
print(f"  num_parameters: {torch_parameter_count(model)}")
print(f"  f1_micro_baseline: {summary_05['f1_micro']:.6f}")
print(f"  f1_macro_baseline: {summary_05['f1_macro']:.6f}")
print(f"  f1_micro_tuned: {summary_tuned['f1_micro']:.6f}")
print(f"  f1_macro_tuned: {summary_tuned['f1_macro']:.6f}")

print("\nOutcome interpretation:")
if summary_tuned["f1_macro"] >= summary_05["f1_macro"]:
    print("  Tuned thresholds improve or match macro-F1 by balancing precision/recall per label.")
else:
    print("  Tuned thresholds did not improve macro-F1; consider narrower search or more training epochs.")
print("  Rare labels often gain recall with lower thresholds, with some precision tradeoff.")

Baseline metrics (threshold=0.5): {'f1_micro': 0.7694864048338369, 'f1_macro': 0.5667263938244894}
Tuned metrics (per-label thresholds): {'f1_micro': 0.7760184057382595, 'f1_macro': 0.6656803604220689}

Per-label chosen thresholds:
{'toxic': 0.36000000000000004, 'severe_toxic': 0.20000000000000004, 'obscene': 0.6600000000000001, 'threat': 0.25000000000000006, 'insult': 0.22000000000000003, 'identity_hate': 0.36000000000000004}


,setting,f1_micro,f1_macro
0,baseline_0.5,0.769486,0.566726
1,tuned_per_label,0.776018,0.665680


,label,best_threshold,best_f1_on_val
0,toxic,0.36,0.827406
1,severe_toxic,0.20,0.525386
2,obscene,0.66,0.836747
3,threat,0.25,0.495575
4,insult,0.22,0.755776
5,identity_hate,0.36,0.553191


,label,precision_baseline,recall_baseline,f1_baseline,roc_auc_baseline,precision_tuned,recall_tuned,f1_tuned,roc_auc_tuned,best_threshold
0,toxic,0.887036,0.759247,0.818182,0.985809,0.832021,0.822842,0.827406,0.985809,0.36
1,severe_toxic,0.679245,0.220859,0.333333,0.991136,0.410345,0.730061,0.525386,0.991136,0.20
2,obscene,0.820513,0.848193,0.834123,0.992251,0.862996,0.812048,0.836747,0.992251,0.66
3,threat,0.384615,0.094340,0.151515,0.992874,0.466667,0.528302,0.495575,0.992874,0.25
4,insult,0.767857,0.713921,0.739907,0.987312,0.663768,0.877395,0.755776,0.987312,0.22
5,identity_hate,0.532847,0.514085,0.523297,0.987285,0.486631,0.640845,0.553191,0.987285,0.36



Confusion matrices (tuned predictions)
toxic 
 [[14161   256]
 [  273  1268]]
severe_toxic 
 [[15624   171]
 [   44   119]]
obscene 
 [[15021   107]
 [  156   674]]
threat 
 [[15873    32]
 [   25    28]]
insult 
 [[14827   348]
 [   96   687]]
identity_hate 
 [[15720    96]
 [   51    91]]

--- Proposal summary (record for report / comparison) ---
  device: mps
  training_time_s: 3376.60
  num_parameters: 66958086
  f1_micro_baseline: 0.769486
  f1_macro_baseline: 0.566726
  f1_micro_tuned: 0.776018
  f1_macro_tuned: 0.665680

Outcome interpretation:
  Tuned thresholds improve or match macro-F1 by balancing precision/recall per label.
  Rare labels often gain recall with lower thresholds, with some precision tradeoff.
